In [1]:
import re
import sys
import json
import csv
from pathlib import Path
import pdfplumber


# ----------------------------------------------------------------------
# Regex: one service block, with an OPTIONAL trailing denial block.
# ----------------------------------------------------------------------
SERVICE_PATTERN = re.compile(
    r'(?P<code>D\d{4})\s*-.*?\n'
    r'Quantity\s+\d+\s+Deductible\s+(?P<deductible>\$[\d,.]+)\s+'
    r'Coinsurance Computed\s+(?P<coins>\$[\d,.]+)\s+'
    r'Billed Amount\s+(?P<billed>\$[\d,.]+)\s*\n'
    r'Service Date\s+(?P<svc_date>\d{2}/\d{2}/\d{4})\s+'
    r'COB Collected\s+(?P<cob>\$[\d,.]+)\s+'
    r'Payable\s+(?P<payable>\$[\d,.]+)\s+'
    r'Allowed Amount\s+(?P<allowed>\$[\d,.]+)\s*\n'
    r'(?:Auth Number)?\s*Copay Computed\s+(?P<copay>\$[\d,.]+)\s+'
    r'Over Maximum\s+(?P<over_max>\$[\d,.]+)\s+'
    r'Paid Amount\s+(?P<paid>\$[\d,.]+)'
    r'(?:\s*\nService Line Denial/Exception.*?following reasons:\s*\n'
    r'(?P<denial_text>(?:(?!\nD\d{4}|\nClaim Status|\nRendered:).)*))?',
    re.DOTALL
)


def extract_claim_pdf(pdf_path):
    """Extract claim information from one PDF across ALL pages."""

    pdf_path = Path(pdf_path)

    with pdfplumber.open(pdf_path) as pdf:

        # Extract text from every page
        page_texts = [p.extract_text() or "" for p in pdf.pages]

        # Find patient/provider header
        patient_words = []
        provider_words = []

        for page, text in zip(pdf.pages, page_texts):

            if "Patient Information" in text:

                words = page.extract_words()

                patient_words = [
                    w for w in words
                    if w["x0"] < 200 and 75 < w["top"] < 85
                ]

                provider_words = [
                    w for w in words
                    if 205 < w["x0"] < 410 and 75 < w["top"] < 85
                ]

                break

    # Combine all pages
    full_text = "\n".join(page_texts)

    data = {
        "file_name": pdf_path.name
    }

    # --------------------------------------------------------------
    # Patient Name
    # --------------------------------------------------------------
    data["patient_name"] = " ".join(
        w["text"] for w in patient_words
    ).strip()

    # --------------------------------------------------------------
    # Provider Name
    # --------------------------------------------------------------
    provider_line = " ".join(
        w["text"] for w in provider_words
    )

    data["provider_name"] = re.sub(
        r"\s*\(\d+\)\s*$",
        "",
        provider_line
    ).strip()

    # --------------------------------------------------------------
    # Claim Status
    # --------------------------------------------------------------
    status_match = re.search(
        r"Claim Status:\s*(\w+)",
        full_text
    )

    data["claim_status"] = (
        status_match.group(1)
        if status_match
        else None
    )

    # --------------------------------------------------------------
    # Service Details
    # --------------------------------------------------------------
    services = []

    for m in SERVICE_PATTERN.finditer(full_text):

        denial_text = m.group("denial_text")

        denial_text = (
            denial_text.strip()
            if denial_text
            else None
        )

        services.append({
            "service_code": m.group("code"),
            "deductible": m.group("deductible"),
            "coinsurance_computed": m.group("coins"),
            "billed_amount": m.group("billed"),
            "service_date": m.group("svc_date"),
            "cob_collected": m.group("cob"),
            "payable": m.group("payable"),
            "allowed_amount": m.group("allowed"),
            "copay_computed": m.group("copay"),
            "over_maximum": m.group("over_max"),
            "paid_amount": m.group("paid"),
            "denial_status": (
                "Denied"
                if denial_text
                else "Not Denied"
            ),
            "denial_reason": denial_text,
        })

    data["services"] = services

    # --------------------------------------------------------------
    # Payment Information / Totals
    # --------------------------------------------------------------
    def find_val(label):

        match = re.search(
            re.escape(label) + r"\s+(\$[\d,.]+)",
            full_text
        )

        return match.group(1) if match else None

    data["totals"] = {
        "total_billed_amount": find_val(
            "Total Billed Amount"
        ),
        "original_paid_amount": find_val(
            "Original Paid Amount"
        ),
        "net_paid_amount": find_val(
            "Net Paid Amount"
        ),
    }

    return data


def flatten_for_table(claim):
    """Create one row per service."""

    base = {
        "file_name": claim["file_name"],
        "patient_name": claim["patient_name"],
        "provider_name": claim["provider_name"],
        "claim_status": claim["claim_status"],
        "total_billed_amount": claim["totals"][
            "total_billed_amount"
        ],
        "original_paid_amount": claim["totals"][
            "original_paid_amount"
        ],
        "net_paid_amount": claim["totals"][
            "net_paid_amount"
        ],
    }

    rows = []

    if claim["services"]:

        for svc in claim["services"]:

            row = dict(base)
            row.update(svc)
            rows.append(row)

    else:

        rows.append(base)

    return rows


def run_pipeline(pdf_path, output_root="./test_1"):
    """
    Process one PDF and save JSON inside:

        test_1/
            PDF_NAME/
                PDF_NAME.json
    """

    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}"
        )

    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(
            "Input file must be a PDF."
        )

    folder_name = pdf_path.stem

    output_folder = (
        Path(output_root) / folder_name
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    json_path = (
        output_folder /
        f"{folder_name}.json"
    )

    print("=" * 70)
    print(f"Processing: {pdf_path.name}")
    print("=" * 70)

    extracted_data = extract_claim_pdf(
        pdf_path
    )

    claim_data = [extracted_data]

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            claim_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    n_denied = sum(
        1
        for s in extracted_data["services"]
        if s["denial_status"] == "Denied"
    )

    print(
        f"Patient      : "
        f"{extracted_data['patient_name']}"
    )

    print(
        f"Provider     : "
        f"{extracted_data['provider_name']}"
    )

    print(
        f"Claim Status : "
        f"{extracted_data['claim_status']}"
    )

    print(
        f"Services     : "
        f"{len(extracted_data['services'])} "
        f"({n_denied} denied)"
    )

    print(
        f"Total Billed : "
        f"{extracted_data['totals']['total_billed_amount']}"
    )

    print(
        f"Original Paid: "
        f"{extracted_data['totals']['original_paid_amount']}"
    )

    print(
        f"Net Paid     : "
        f"{extracted_data['totals']['net_paid_amount']}"
    )

    print()

    print("JSON saved to:")
    print(json_path)

    print("=" * 70)

    return claim_data, json_path


def run_batch(input_dir, output_path):
    """
    Process every PDF in a folder and create:

    1. Excel/CSV table
    2. Combined JSON
    """

    input_dir = Path(input_dir)
    output_path = Path(output_path)

    pdf_files = sorted(
        input_dir.glob("*.pdf")
    )

    if not pdf_files:

        print(
            f"No PDF files found in {input_dir}"
        )

        return

    all_rows = []
    all_claims = []
    errors = []

    for pdf_path in pdf_files:

        try:

            claim = extract_claim_pdf(
                pdf_path
            )

            all_claims.append(claim)

            all_rows.extend(
                flatten_for_table(claim)
            )

            n_services = len(
                claim["services"]
            )

            n_denied = sum(
                1
                for s in claim["services"]
                if s["denial_status"] == "Denied"
            )

            flag = (
                ""
                if n_services
                else
                "  <-- NO SERVICE ROWS MATCHED, CHECK THIS FILE"
            )

            print(
                f"OK   {pdf_path.name}: "
                f"{n_services} service(s), "
                f"{n_denied} denied, "
                f"status={claim['claim_status']}"
                f"{flag}"
            )

        except Exception as e:

            errors.append(
                (pdf_path.name, str(e))
            )

            print(
                f"FAIL {pdf_path.name}: {e}"
            )

    columns = [
        "file_name",
        "patient_name",
        "provider_name",
        "claim_status",
        "service_code",
        "deductible",
        "coinsurance_computed",
        "billed_amount",
        "service_date",
        "cob_collected",
        "payable",
        "allowed_amount",
        "copay_computed",
        "over_maximum",
        "paid_amount",
        "denial_status",
        "denial_reason",
        "total_billed_amount",
        "original_paid_amount",
        "net_paid_amount",
    ]

    # --------------------------------------------------------------
    # Excel output
    # --------------------------------------------------------------
    if output_path.suffix.lower() == ".xlsx":

        import openpyxl
        from openpyxl.utils import get_column_letter

        wb = openpyxl.Workbook()

        ws = wb.active
        ws.title = "Claims"

        ws.append(columns)

        for row in all_rows:

            ws.append([
                row.get(c, "")
                for c in columns
            ])

        for i, col in enumerate(
            columns,
            start=1
        ):

            max_len = max(
                [len(col)] +
                [
                    len(str(r.get(col, "")))
                    for r in all_rows
                ]
            )

            ws.column_dimensions[
                get_column_letter(i)
            ].width = min(
                max_len + 2,
                40
            )

        wb.save(output_path)

    # --------------------------------------------------------------
    # CSV output
    # --------------------------------------------------------------
    else:

        with open(
            output_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=columns
            )

            writer.writeheader()

            for row in all_rows:
                writer.writerow(row)

    # --------------------------------------------------------------
    # Combined JSON
    # --------------------------------------------------------------
    json_path = output_path.with_suffix(
        ".json"
    )

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            all_claims,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"\nDone. "
        f"{len(pdf_files)} PDF(s) processed, "
        f"{len(errors)} failed."
    )

    print(
        f"Table written to:  {output_path}"
    )

    print(
        f"Raw JSON written to: {json_path}"
    )

    if errors:

        print("\nFiles that failed:")

        for name, err in errors:

            print(
                f"  - {name}: {err}"
            )

In [2]:
pdf_path = r"/home/cipl/users/Jeeva/Phase_2_pdf/Highmark Wholecare/DELVALLE, TOMAS.pdf"

claim_data, json_path = run_pipeline(
    pdf_path,
    output_root="./test_1"
)

Processing: DELVALLE, TOMAS.pdf
Patient      : Tomas Delvalle
Provider     : Kaushal Patel
Claim Status : Processed
Services     : 2 (2 denied)
Total Billed : $140.00
Original Paid: $0.00
Net Paid     : $0.00

JSON saved to:
test_1/DELVALLE, TOMAS/DELVALLE, TOMAS.json
